In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/DeepLearningProject/data/train/task2train540p /content/

Basic CNN Structure
```
Image
  ↓
Conv Layer
  ↓
ReLU
  ↓
MaxPool
  ↓
Conv Layer
  ↓
ReLU
  ↓
MaxPool
  ↓
Flatten
  ↓
Fully Connected Layer
  ↓
Output (4 classes)

```



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class TrafficSignCNN(nn.Module):

    def __init__(self, num_classes=4):
        super(TrafficSignCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(32 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))

        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

> **nn.Module**

In PyTorch, every neural network must inherit from nn.Module because:

It stores parameters
, It enables backpropagation
, It connects model to optimizer
, It allows .to(device) for GPU
, It enables .eval() and .train()

---
**__init__()** is the constructor of the class.

Defines layers
, Initializes weights
, Builds model architecture
, This runs only once when the model is created.

---

```
# self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
```

A convolution layer applies filters (kernels) over the image.

**in_channels=3**	- RGB image

**out_channels=16**	- 16 feature maps

**kernel_size=3**	- 3×3 filter

**padding=1**	- keeps spatial size same

---
Output = ((W - K + 2 * P) / S) + 1

Where:

W = input size
,K = kernel size
,P = padding
,S = stride

---
```
self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
```
> **What is Max Pooling?**

It reduces spatial size by taking max value in region.

**Why use pooling?**

Reduces computation
, Reduces overfitting
, Makes model translation invariant
, Keeps important features

**Effect:**

Before: 128 x 128
After: 64 x 64

Output = ((W - K) / S) + 1

---
```
self.conv2 = nn.Conv2d(16,32,kernel_size=3, padding=1)
```
Meaning:

16 - input channels from conv1

32 - output feature maps

**Why increase channels?**

**Early layers detect:**

1. edges

2. colors

**Deeper layers detect:**

1. shapes

2. complex patterns

3. sign symbols

More channels = more feature detectors

---

```
self.fc1 = nn.Linear(32*32*32,128)
self.fc2 = nn.Linear(128,num_classes)
```

> **Why Fully Connected?**

Convolution extracts features.

Fully connected performs classification.

* Input : 128×128

* After conv1 : 128×128

* After pool : 64×64

* After conv2 : 64×64

* After pool : 32×32

Channels after conv2 = 32

***So tensor shape:***

```[Batch, 32, 32, 32]```

***Flatten :***

```32 × 32 × 32 = 32768```

***That is why:***

```nn.Linear(32*32*32, 128)```
---



> Forward Function

```
def forward(self,x):
```

Why forward function?

Defines:

* Data flow

* Layer execution order

PyTorch automatically calls forward() when you do


```
 output = model(input)
```
---
```
x = self.pool(F.relu(self.conv1(x)))
```
> **ReLU Activation**

```F.relu()```

***ReLU(x)=max(0,x)***

> Why ReLU?

* Adds non-linearity

* Prevents vanishing gradient

* Makes training faster

**Without activation:**

→ Network becomes linear

→ Cannot learn complex patterns

---

> Second Block
```
x = self.pool(F.relu(self.conv2(x)))
```
Conv → ReLU → Pool

---
> Flatten

Why flatten?

Fully connected layer expects:

```[Batch, Features]```

Not : ```[Batch, Channels, Height, Width]```

**So we convert :**
```
[Batch, 32, 32, 32]
        ↓
[Batch, 32768]
```

---

> Final layers :

```
x = F.relu(self.fc1(x))
x = self.fc2(x)
```

**fc1:**

* Reduces dimension

* Learns high-level representation

**fc2:**

* Outputs class scores

If num_classes = 4

Output shape:
[Batch, 4]

Each value = class score (logit)

```
Input Image (3x128x128)
        ↓
Conv1 (16 filters)
        ↓
      ReLU
        ↓
    MaxPool
        ↓
    (16x64x64)
        ↓
Conv2 (32 filters)
        ↓
      ReLU
        ↓
      MaxPool
        ↓
    (32x32x32)
        ↓
    Flatten
        ↓
    (32768)
        ↓
    FC1 (128)
        ↓
      ReLU
        ↓
FC2 (4 classes)
        ↓
      Output
```

Conv1 :	Detect edges & textures

Pool1	: Reduce size

Conv2	: Detect shapes & symbols

Pool2	: Reduce size

FC1	: Combine features

FC2	: Classify

Why __init__()?

* To define model layers.

Why forward()?

* To define how data flows.

Why ReLU?

* To add non-linearity.

Why Pooling?

* To reduce size and overfitting.

Why Fully Connected?

* To perform classification.

Why Flatten?

* To convert 3D feature maps to 1D vector.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TrafficSignCNN(num_classes=4)
model = model.to(device)

print(model)

TrafficSignCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=4, bias=True)
)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

>What is CrossEntropyLoss?

Used for multi-class classification.

It:

* Combines Softmax

* Negative Log Likelihood

You DO NOT apply Softmax manually.

> What is Adam Optimizer?

Adam = Adaptive Moment Estimation

It:

* Adjusts learning rate automatically

* Converges faster

* Works well in most cases